# Talk To Data - Educational Notebook

**Learn AI-powered natural language to SQL**

This notebook demonstrates how to build a system that converts plain English questions into SQL queries using large language models (LLMs). Each cell represents one key concept in the pipeline.

## What You'll Learn
1. Load and explore data
2. Generate structured data semantics using AI
3. Define business logic and terminology
4. Provide reference queries for few-shot learning
5. Build a query engine that generates SQL from natural language
6. Execute queries and display results

## Prerequisites
- Python 3.9+
- OpenAI API key (or Anthropic API key)
- Basic SQL knowledge

## Setup
If running in Google Colab:
1. Go to **Secrets** (key icon in left sidebar)
2. Add secret: `OPENAI_API_KEY` with your API key
3. Run all cells below

## Cell 1: Setup & Imports

First, we install dependencies and configure our environment.

In [ ]:
# Install dependencies (only needed in Colab or fresh environments)
!pip install -q openai pandas pyyaml

import os
import sqlite3
import pandas as pd
import yaml
from openai import OpenAI

# Configure API key
# For Colab: use userdata.get('OPENAI_API_KEY')
# For local: set environment variable OPENAI_API_KEY
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except ImportError:
    api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError("API key not found. Set OPENAI_API_KEY in Colab Secrets or environment.")

# Initialize OpenAI client
client = OpenAI(api_key=api_key)

print("✅ Setup complete!")

## Cell 2: Data Ingestion

Load sample datasets (customers, orders, products) into pandas DataFrames, then create an in-memory SQLite database.

In [ ]:
# Download sample data from GitHub (if running in Colab)
base_url = "https://raw.githubusercontent.com/dave-melillo/talk-to-data-notebook/main/data/"

try:
    # Try to load from local files first
    df_customers = pd.read_csv('data/customers.csv')
    df_orders = pd.read_csv('data/orders.csv')
    df_products = pd.read_csv('data/products.csv')
except FileNotFoundError:
    # Fallback to GitHub
    df_customers = pd.read_csv(base_url + 'customers.csv')
    df_orders = pd.read_csv(base_url + 'orders.csv')
    df_products = pd.read_csv(base_url + 'products.csv')

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')
df_customers.to_sql('customers', conn, index=False, if_exists='replace')
df_orders.to_sql('orders', conn, index=False, if_exists='replace')
df_products.to_sql('products', conn, index=False, if_exists='replace')

# Display sample data
print("📊 Customers Table:")
display(df_customers.head())

print("\n📦 Orders Table:")
display(df_orders.head())

print("\n🛍️ Products Table:")
display(df_products.head())

print(f"\n✅ Loaded {len(df_customers)} customers, {len(df_orders)} orders, {len(df_products)} products")

## Cell 3: Data Semantic Generation (AI)

Use an LLM to analyze the database schema and generate structured metadata about each table and column. This helps the query engine understand what the data represents.

In [ ]:
def generate_data_semantic(df_dict):
    """Generate structured data semantic using LLM."""
    
    # Build schema description
    schema_info = {}
    for table_name, df in df_dict.items():
        schema_info[table_name] = {
            'columns': list(df.columns),
            'dtypes': df.dtypes.astype(str).to_dict(),
            'sample_rows': df.head(3).to_dict('records')
        }
    
    # LLM prompt
    prompt = f"""Analyze this database schema and generate a structured YAML description.

Schema:
{yaml.dump(schema_info, default_flow_style=False)}

Generate YAML output with this structure:
```yaml
tables:
  table_name:
    description: "Human-readable description"
    columns:
      column_name:
        type: "data type"
        description: "What this column represents"
        primary_key: true/false
    relationships:
      - foreign_key: column_name
        references: other_table.column_name
```

Return ONLY the YAML, no explanations."""
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    yaml_text = response.choices[0].message.content.strip()
    # Remove markdown code fences if present
    if yaml_text.startswith('```yaml'):
        yaml_text = yaml_text.split('```yaml')[1].split('```')[0].strip()
    elif yaml_text.startswith('```'):
        yaml_text = yaml_text.split('```')[1].split('```')[0].strip()
    
    return yaml.safe_load(yaml_text)

# Generate data semantic
data_semantic = generate_data_semantic({
    'customers': df_customers,
    'orders': df_orders,
    'products': df_products
})

print("🧠 Generated Data Semantic:")
print(yaml.dump(data_semantic, default_flow_style=False))

print("✅ Data semantic generated!")

## Cell 4: Business Semantic (User Input)

Define business-specific terminology, KPIs, and rules. This helps the AI understand domain-specific concepts like "churned customer" or "lifetime value."

In [ ]:
# Business semantic template (user-editable)
business_semantic = {
    'glossary': {
        'churned_customer': "A customer with status = 'churned'",
        'active_customer': "A customer with status = 'active'",
        'high_value_customer': "A customer with lifetime_value > 10000",
        'recent_order': "An order placed in the last 30 days"
    },
    'kpis': {
        'total_revenue': 'SUM(orders.order_total)',
        'average_order_value': 'AVG(orders.order_total)',
        'customer_count': 'COUNT(DISTINCT customers.customer_id)',
        'total_orders': 'COUNT(orders.order_id)'
    },
    'rules': [
        "Always join orders to customers via customer_id",
        "Always join orders to products via product_id",
        "Use created_at for customer signup date",
        "Use order_date for order timing analysis"
    ]
}

print("📝 Business Semantic:")
print(yaml.dump(business_semantic, default_flow_style=False))

print("✅ Business semantic defined!")

## Cell 5: Reference Queries (Few-Shot Learning)

Provide example question-SQL pairs to guide the LLM. These examples help the AI learn your preferred SQL style and query patterns.

In [ ]:
# Reference queries for few-shot learning
reference_queries = [
    {
        'question': 'Who are my top 10 customers by lifetime value?',
        'sql': 'SELECT customer_id, name, lifetime_value FROM customers ORDER BY lifetime_value DESC LIMIT 10',
        'explanation': 'Sort customers by lifetime_value descending and limit to 10'
    },
    {
        'question': 'What is the total revenue for March 2024?',
        'sql': "SELECT SUM(order_total) as total_revenue FROM orders WHERE order_date >= '2024-03-01' AND order_date < '2024-04-01'",
        'explanation': 'Filter orders by date range and sum order_total'
    },
    {
        'question': 'Which products have low stock (less than 100 units)?',
        'sql': 'SELECT product_id, name, stock_quantity FROM products WHERE stock_quantity < 100 ORDER BY stock_quantity ASC',
        'explanation': 'Filter products where stock is below threshold'
    },
    {
        'question': 'How many active customers do we have?',
        'sql': "SELECT COUNT(*) as active_customers FROM customers WHERE status = 'active'",
        'explanation': 'Count customers with active status'
    },
    {
        'question': 'What are the best-selling products by quantity?',
        'sql': '''SELECT p.product_id, p.name, SUM(o.quantity) as total_sold
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_id, p.name
ORDER BY total_sold DESC
LIMIT 10''',
        'explanation': 'Join orders and products, sum quantity, group by product'
    }
]

print("📚 Reference Queries:")
for i, ref in enumerate(reference_queries, 1):
    print(f"\n{i}. {ref['question']}")
    print(f"   SQL: {ref['sql'][:80]}..." if len(ref['sql']) > 80 else f"   SQL: {ref['sql']}")

print(f"\n✅ Loaded {len(reference_queries)} reference queries!")

## Cell 6: Query Engine (AI SQL Generation)

Combine data semantic + business semantic + reference queries to generate SQL from natural language questions.

In [ ]:
def ask_question(question: str, show_prompt=False) -> str:
    """Convert natural language question to SQL query."""
    
    # Build context for LLM
    context = f"""You are a SQL expert. Convert natural language questions to SQL queries.

DATABASE SCHEMA:
{yaml.dump(data_semantic, default_flow_style=False)}

BUSINESS RULES:
{yaml.dump(business_semantic, default_flow_style=False)}

EXAMPLE QUERIES:
"""
    for ref in reference_queries:
        context += f"\nQ: {ref['question']}\nSQL: {ref['sql']}\n"
    
    context += f"""\nRULES:
1. Return ONLY the SQL query, no explanations
2. Use SQLite syntax
3. Follow the patterns in example queries
4. Use proper JOINs when querying multiple tables
5. Use meaningful column aliases

USER QUESTION: {question}
SQL:"""
    
    if show_prompt:
        print("🔍 Full Prompt Sent to LLM:")
        print("="*80)
        print(context)
        print("="*80)
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": context}],
        temperature=0
    )
    
    sql = response.choices[0].message.content.strip()
    
    # Clean up response (remove markdown formatting if present)
    if sql.startswith('```sql'):
        sql = sql.split('```sql')[1].split('```')[0].strip()
    elif sql.startswith('```'):
        sql = sql.split('```')[1].split('```')[0].strip()
    
    return sql

# Test the query engine
test_question = "Who are my top 5 customers by total spending?"
generated_sql = ask_question(test_question)

print(f"❓ Question: {test_question}")
print(f"\n🤖 Generated SQL:")
print(generated_sql)

print("\n✅ Query engine ready!")

## Cell 7: Execute Query & Display Results

Execute the generated SQL and display results as a pandas DataFrame.

In [ ]:
def execute_query(sql: str) -> pd.DataFrame:
    """Execute SQL query and return results as DataFrame."""
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        print(f"❌ SQL Error: {e}")
        print(f"\nGenerated SQL:\n{sql}")
        return None

# Execute the test query
results = execute_query(generated_sql)

if results is not None:
    print("📊 Query Results:")
    display(results)
    print(f"\n✅ Returned {len(results)} rows")
else:
    print("⚠️ Query failed. Check SQL syntax.")

## Cell 8: Interactive Query Interface

Now you can ask your own questions! Try these examples:
- "What were the total sales in March 2024?"
- "Which customers haven't ordered in the last 30 days?"
- "What is the average profit margin by product category?"
- "Who are the churned customers?"
- "What are the top 10 best-selling products?"

In [ ]:
# Interactive query function
def query(question: str, show_sql=True, show_prompt=False):
    """Ask a question and get results."""
    print(f"❓ Question: {question}\n")
    
    # Generate SQL
    sql = ask_question(question, show_prompt=show_prompt)
    
    if show_sql:
        print("🤖 Generated SQL:")
        print(sql)
        print()
    
    # Execute query
    results = execute_query(sql)
    
    if results is not None:
        print("📊 Results:")
        display(results)
        print(f"\n✅ {len(results)} rows returned")
    
    return results

# Example usage
query("What are the top 10 best-selling products by revenue?")

## Try Your Own Questions!

Modify the cell below to ask your own questions:

In [ ]:
# Ask your own question here
your_question = "How many orders were placed by high-value customers?"

query(your_question)

## Advanced: Debug Mode

See the full prompt sent to the LLM by setting `show_prompt=True`:

In [ ]:
query("What is the average order value by customer status?", show_prompt=True)

## What's Next?

**Experiment with:**
- Different datasets (upload your own CSV files)
- Modified business semantics (add new KPIs)
- Additional reference queries (teach new patterns)
- Different LLM models (Claude, Gemini, local models)

**Learn more:**
- [Production version: talk-to-data](https://github.com/dave-melillo/talk-to-data)
- [OpenAI API Documentation](https://platform.openai.com/docs)
- [SQLite Tutorial](https://www.sqlitetutorial.net/)

---

**Built with ❤️ for learning and teaching.**